<a href="https://colab.research.google.com/github/willy410-hub/extraction-sft-rl/blob/main/notebooks/try_extraction_sft_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Try extraction-sft-rl: input, model, output

This notebook runs a small open model (Qwen2.5-0.5B-Instruct) on messy, invoice-like messages and checks its JSON answer against the correct one.

All data is synthetic. Project: https://github.com/willy410-hub/extraction-sft-rl

**Faster (optional):** Runtime, then Change runtime type, then pick **T4 GPU**. It also works on CPU, just slower.

Run the cells from top to bottom (Shift + Enter).

## 1. Download the project and install what it needs
If Colab asks you to restart the session after this cell, restart it and continue with step 2.

In [ ]:
!git clone -q https://github.com/willy410-hub/extraction-sft-rl
!pip install -q "transformers==5.21.0" "trl==1.16.0" "peft==0.21.2" "datasets==5.1.0" "accelerate==1.15.0"

## 1b. Run the tests (no model download needed)

In [ ]:
!cd /content/extraction-sft-rl && python -m pytest -q

## 2. Look at the data
Each example is a messy message plus the correct answer. The answer is chosen first and then written into the text, so we always know what is right.

In [ ]:
%cd /content/extraction-sft-rl
from extraction_rl.generator import make_split

examples = make_split("test", 4)   # change 4 to see more
for e in examples:
    print("NOISE TYPE:", e["noise"])
    print(e["text"])
    print("CORRECT ANSWER:", e["truth"])
    print("-" * 60)

## 3. Run the model and grade its answers
The first run downloads the model (about 1 GB). Score 1.0 means all 6 fields are right.
**Note:** this is the untuned base model, before any training. On the 400-item test split it gets 0.345 exact match; the SFT model in the repo gets 0.97 to 0.995. The trained adapter is not published yet, so mistakes below are expected.

In [ ]:
from extraction_rl.inference import load_model, generate
from extraction_rl.verifier import verify

model, tok = load_model()
outputs, _ = generate(model, tok, [e["text"] for e in examples])

for e, out in zip(examples, outputs):
    result = verify(out, e["truth"])
    print("INPUT:\n" + e["text"])
    print("MODEL OUTPUT:", out)
    print("CORRECT ANSWER:", e["truth"])
    print("SCORE:", result["score"], "| fields right:", result["per_field"])
    print("-" * 60)

## 4. Try your own message
Write any invoice-like text below. There is no correct answer here, so you just see what the model returns.

In [ ]:
my_text = """Hello, please pay invoice INV-12345 for customer Ana Peña (ID C-2024),
total 1,250.00 EUR, dated 3 March 2026."""

out, _ = generate(model, tok, [my_text])
print(out[0])